In [1]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        ssl_ca=os.getenv("DB_SSL_CA")
    )

conn = get_connection()
conn.close()

In [2]:
import pandas as pd
import google.genai as genai
from tabulate import tabulate
from IPython.display import display


In [3]:
from google import genai

client = genai.Client(api_key=os.getenv("LLM_API_KEY"))

In [4]:
SYSTEM_PROMPT = """
Sos un experto en bases de datos MySQL. Tu única tarea es convertir preguntas 
en español a consultas SQL válidas para la base de datos 'biblioia'.

REGLAS:
- Respondé ÚNICAMENTE con la consulta SQL, sin explicaciones ni comentarios.
- No uses bloques de código ni backticks.
- Si la pregunta no se puede responder con el esquema dado, respondé: 
  SELECT 'No puedo responder esa pregunta con los datos disponibles';

=== ESQUEMA ===

GENERO (id_genero INT PK, nombre VARCHAR(60) UNIQUE NOT NULL, descripcion VARCHAR(255))

AUTOR (id_autor INT PK, nombre VARCHAR(80) NOT NULL, apellido VARCHAR(80) NOT NULL, nacionalidad VARCHAR(60))

LIBRO (isbn VARCHAR(20) PK, titulo VARCHAR(200) NOT NULL, anio_publicacion YEAR,
       stock_total SMALLINT, stock_disponible SMALLINT)
  -- stock_disponible <= stock_total siempre
  -- stock_disponible >= 0 siempre

LIBRO_AUTOR (isbn FK->LIBRO, id_autor FK->AUTOR) -- relación N:M entre LIBRO y AUTOR

LIBRO_GENERO (isbn FK->LIBRO, id_genero FK->GENERO) -- relación N:M entre LIBRO y GENERO

SOCIO (id_socio INT PK, dni VARCHAR(15) UNIQUE, nombre VARCHAR(80), apellido VARCHAR(80),
       email VARCHAR(120) UNIQUE, fecha_alta DATE, estado VARCHAR(12))
  -- estado puede ser: 'ACTIVO', 'SUSPENDIDO', 'BAJA'

EJEMPLAR (id_ejemplar INT PK, isbn FK->LIBRO, nro_ejemplar SMALLINT, estado_fisico VARCHAR(12))
  -- estado_fisico puede ser: 'BUENO', 'DETERIORADO', 'BAJA'

PRESTAMO (id_prestamo INT PK, id_socio FK->SOCIO, id_ejemplar FK->EJEMPLAR,
          fecha_prestamo DATE, fecha_vencimiento DATE, fecha_devolucion DATE NULL, estado VARCHAR(12))
  -- estado puede ser: 'ACTIVO', 'DEVUELTO', 'VENCIDO'
  -- fecha_devolucion es NULL mientras el préstamo está activo

SANCION (id_sancion INT PK, id_socio FK->SOCIO, tipo VARCHAR(20), fecha_inicio DATE, fecha_fin DATE, motivo VARCHAR(255))
  -- tipo puede ser: 'MORA', 'DAÑO', 'PERDIDA', 'OTRO'
  -- una sanción está activa cuando fecha_fin >= CURRENT_DATE

AUDITORIA_PRESTAMOS (id_audit INT PK, id_prestamo INT, operacion VARCHAR(10),
                     estado_nuevo VARCHAR(12), estado_viejo VARCHAR(12), usuario_bd VARCHAR(80), fecha_hora DATETIME)
=== EJEMPLOS ===

Pregunta: ¿Cuáles son los 5 libros más prestados este año?
SQL: SELECT L.isbn, L.titulo, COUNT(P.id_prestamo) AS cantidad_prestamos FROM LIBRO L JOIN EJEMPLAR E ON L.isbn = E.isbn JOIN PRESTAMO P ON E.id_ejemplar = P.id_ejemplar WHERE YEAR(P.fecha_prestamo) = YEAR(CURRENT_DATE) GROUP BY L.isbn, L.titulo ORDER BY cantidad_prestamos DESC LIMIT 5;

Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
SQL: SELECT DISTINCT S.id_socio, S.dni, S.nombre, S.apellido FROM SOCIO S JOIN PRESTAMO P ON S.id_socio = P.id_socio WHERE P.estado = 'ACTIVO' AND P.fecha_vencimiento < CURRENT_DATE;

Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
SQL: SELECT L.isbn, L.titulo, L.stock_disponible FROM LIBRO L JOIN LIBRO_GENERO LG ON L.isbn = LG.isbn JOIN GENERO G ON LG.id_genero = G.id_genero WHERE G.nombre LIKE '%ciencia ficción%' AND L.stock_disponible > 0;
"""


In [5]:
def text_to_sql(pregunta: str) -> str:
    respuesta = client.models.generate_content(
       model="gemini-2.5-flash",
        contents=SYSTEM_PROMPT + f"\nPregunta: {pregunta}\nSQL:"
    )
    sql = respuesta.text.strip()
    # Por las dudas, limpiamos backticks que Gemini a veces agrega
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


In [6]:
def ejecutar_consulta(sql: str) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})
    # cirra la conexion 
    finally:
        conn.close()


In [7]:
def agente_responder(pregunta: str, mostrar_sql: bool = True):
    print(f"\n Pregunta: {pregunta}")
    print("-" * 60)
    
    sql = text_to_sql(pregunta)
    
    if mostrar_sql:
        print(f" SQL generado:\n{sql}")
        print("-" * 60)
    
    df = ejecutar_consulta(sql)
    
    if df.empty:
        print(" La consulta no devolvió resultados.")
    else:
        print(f" Resultado ({len(df)} filas):")
        display(df)
    
    return df

In [8]:
agente_responder("¿Cuáles son los 5 libros más prestados este año?")


 Pregunta: ¿Cuáles son los 5 libros más prestados este año?
------------------------------------------------------------
 SQL generado:
SELECT L.isbn, L.titulo, COUNT(P.id_prestamo) AS cantidad_prestamos FROM LIBRO L JOIN EJEMPLAR E ON L.isbn = E.isbn JOIN PRESTAMO P ON E.id_ejemplar = P.id_ejemplar WHERE YEAR(P.fecha_prestamo) = YEAR(CURRENT_DATE) GROUP BY L.isbn, L.titulo ORDER BY cantidad_prestamos DESC LIMIT 5;
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (5 filas):


,isbn,titulo,cantidad_prestamos
0,978-0-09-954793-8,Sapiens,4
1,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,3
2,978-0-385-33348-1,21 lecciones para el siglo XXI,3
3,978-0-13-235088-4,Clean Code,3
4,978-950-724-528-3,El Aleph,3


,isbn,titulo,cantidad_prestamos
0,978-0-09-954793-8,Sapiens,4
1,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,3
2,978-0-385-33348-1,21 lecciones para el siglo XXI,3
3,978-0-13-235088-4,Clean Code,3
4,978-950-724-528-3,El Aleph,3


In [9]:
agente_responder("Qué socios tienen préstamos vencidos en este momento?")


 Pregunta: Qué socios tienen préstamos vencidos en este momento?
------------------------------------------------------------
 SQL generado:
SELECT DISTINCT S.id_socio, S.dni, S.nombre, S.apellido FROM SOCIO S JOIN PRESTAMO P ON S.id_socio = P.id_socio WHERE P.estado = 'ACTIVO' AND P.fecha_vencimiento < CURRENT_DATE;
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (7 filas):


,id_socio,dni,nombre,apellido
0,1,30100001,Lucas,Fernández
1,2,30100002,Valentina,Gómez
2,3,30100003,Matías,Rodríguez
3,4,30100004,Sofía,López
4,6,30100006,Camila,Sánchez
5,7,30100007,Nicolás,Pérez
6,16,30100016,Antonella,Jiménez


,id_socio,dni,nombre,apellido
0,1,30100001,Lucas,Fernández
1,2,30100002,Valentina,Gómez
2,3,30100003,Matías,Rodríguez
3,4,30100004,Sofía,López
4,6,30100006,Camila,Sánchez
5,7,30100007,Nicolás,Pérez
6,16,30100016,Antonella,Jiménez


In [12]:
agente_responder("¿Cuántos ejemplares disponibles hay del libro con ISBN 978-0-13-235088-4?")


 Pregunta: ¿Cuántos ejemplares disponibles hay del libro con ISBN 978-0-13-235088-4?
------------------------------------------------------------
 SQL generado:
SELECT stock_disponible FROM LIBRO WHERE isbn = '978-0-13-235088-4';
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (1 filas):


,stock_disponible
0,3


,stock_disponible
0,3


In [13]:
agente_responder("¿Qué libros de ciencia ficción están disponibles para prestar?")


 Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
------------------------------------------------------------
 SQL generado:
SELECT L.isbn, L.titulo, L.stock_disponible FROM LIBRO L JOIN LIBRO_GENERO LG ON L.isbn = LG.isbn JOIN GENERO G ON LG.id_genero = G.id_genero WHERE G.nombre LIKE '%ciencia ficción%' AND L.stock_disponible > 0;
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (7 filas):


,isbn,titulo,stock_disponible
0,978-0-06-112008-4,La mano izquierda de la oscuridad,1
1,978-0-14-118776-1,Los desposeídos,1
2,978-0-345-34314-2,2001: Una odisea del espacio,2
3,978-0-345-39180-3,Cita con Rama,2
4,978-0-553-29335-7,Fundación,2
5,978-0-553-38316-3,El fin de la eternidad,2
6,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,1


,isbn,titulo,stock_disponible
0,978-0-06-112008-4,La mano izquierda de la oscuridad,1
1,978-0-14-118776-1,Los desposeídos,1
2,978-0-345-34314-2,2001: Una odisea del espacio,2
3,978-0-345-39180-3,Cita con Rama,2
4,978-0-553-29335-7,Fundación,2
5,978-0-553-38316-3,El fin de la eternidad,2
6,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,1


In [14]:
agente_responder("¿Cuál es el historial de préstamos del socio con DNI 30100001?")


 Pregunta: ¿Cuál es el historial de préstamos del socio con DNI 30100001?
------------------------------------------------------------
 SQL generado:
SELECT L.titulo, P.fecha_prestamo, P.fecha_vencimiento, P.fecha_devolucion, P.estado FROM SOCIO S JOIN PRESTAMO P ON S.id_socio = P.id_socio JOIN EJEMPLAR E ON P.id_ejemplar = E.id_ejemplar JOIN LIBRO L ON E.isbn = L.isbn WHERE S.dni = '30100001';
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (3 filas):


,titulo,fecha_prestamo,fecha_vencimiento,fecha_devolucion,estado
0,Fundación,2026-05-15,2026-05-29,None,ACTIVO
1,Fundación,2026-01-05,2026-01-19,2026-01-18,DEVUELTO
2,El fin de la eternidad,2026-04-15,2026-04-29,None,VENCIDO


,titulo,fecha_prestamo,fecha_vencimiento,fecha_devolucion,estado
0,Fundación,2026-05-15,2026-05-29,None,ACTIVO
1,Fundación,2026-01-05,2026-01-19,2026-01-18,DEVUELTO
2,El fin de la eternidad,2026-04-15,2026-04-29,None,VENCIDO


In [15]:
agente_responder("¿Qué autores tienen más de 3 libros en la biblioteca?")


 Pregunta: ¿Qué autores tienen más de 3 libros en la biblioteca?
------------------------------------------------------------
 SQL generado:
SELECT A.nombre, A.apellido, COUNT(LA.isbn) AS cantidad_libros FROM AUTOR A JOIN LIBRO_AUTOR LA ON A.id_autor = LA.id_autor GROUP BY A.id_autor, A.nombre, A.apellido HAVING COUNT(LA.isbn) > 3;
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 La consulta no devolvió resultados.


,nombre,apellido,cantidad_libros


In [16]:
agente_responder("¿Cuántas sanciones se generaron en el último mes?")


 Pregunta: ¿Cuántas sanciones se generaron en el último mes?
------------------------------------------------------------
 SQL generado:
SELECT COUNT(id_sancion) FROM SANCION WHERE fecha_inicio >= DATE_SUB(CURRENT_DATE, INTERVAL 1 MONTH);
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_8684\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (1 filas):


,COUNT(id_sancion)
0,0


,COUNT(id_sancion)
0,0
